# 🦿 G1 ヒューマノイドロボットの歩行を学習させよう！

このノートブックでは Unitree **G1** — 29 関節を持つ二足歩行ロボット — に **自力で歩く** ことを学習させます。

## Go2（四足）との違い

| 項目 | Go2 (四足) | **G1 (二足)** |
|------|-----------|---------------|
| 関節数 | 12 | **29** |
| 脚の本数 | 4 | **2** |
| 重心高さ | ~0.3 m | **~0.8 m** |
| 転倒リスク | 低い | **高い** |
| 観測次元 | 45 | **98** |

二足歩行は四足歩行より格段に難しい！ でも報酬を上手に設計すれば、AI は自然な歩き方を習得できます。

## 参考文献
- [unitree_rl_mjlab](https://github.com/unitreerobotics/unitree_rl_mjlab) — Unitree 公式の MuJoCo RL フレームワーク
- [MuJoCo Menagerie G1](https://github.com/google-deepmind/mujoco_menagerie/tree/main/unitree_g1) — G1 MuJoCo モデル

## 流れ
1. 🔧 セットアップ（G1 モデルを取得）
2. 📚 G1 の関節構造を理解する
3. 🎛️ 報酬パラメータを設定する
4. 🚀 歩行を学習させる
5. 🎬 結果を動画で確認する
6. 💾 モデルを保存する

In [ ]:
#@title 🔧 セットアップ（最初に一度だけ実行してください）

import subprocess, sys, os

# ── 1. レンダリングバックエンド設定 ────────────────────────────────────────
_has_gpu = subprocess.run('nvidia-smi', shell=True, capture_output=True).returncode == 0
if _has_gpu:
    _ICD = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
    os.makedirs(os.path.dirname(_ICD), exist_ok=True)
    if not os.path.exists(_ICD):
        with open(_ICD, 'w') as _f:
            _f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}\n')
    os.environ['MUJOCO_GL'] = 'egl'
    print('🖥️  GPU 検出: EGL レンダリング')
else:
    subprocess.run('apt-get install -y -q libosmesa6', shell=True, check=False)
    os.environ['MUJOCO_GL'] = 'osmesa'
    print('💻  CPU モード: OSMesa レンダリング')

# ffmpeg
subprocess.run(
    'command -v ffmpeg >/dev/null || (apt-get update -qq && apt-get install -y -q ffmpeg)',
    shell=True, check=False)

# ── 2. ライブラリインストール ───────────────────────────────────────────────
print('\nライブラリをインストール中...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'mujoco', 'gymnasium', 'stable-baselines3[extra]',
    'mediapy', 'tqdm', 'matplotlib', 'pandas'], check=True)

# ── 3. RoboQuest2026 リポジトリ ────────────────────────────────────────────
if not os.path.exists('/content/RoboQuest2026/.git'):
    print('\nRoboQuest2026 をダウンロード中...')
    subprocess.run(['git', 'clone', '-q',
        'https://github.com/SingularityBattleQuest/RoboQuest2026.git',
        '/content/RoboQuest2026'], check=True)
else:
    subprocess.run(['git', '-C', '/content/RoboQuest2026', 'pull', 'origin', 'main', '-q'], check=False)

os.chdir('/content/RoboQuest2026')
if '/content/RoboQuest2026' not in sys.path:
    sys.path.insert(0, '/content/RoboQuest2026')

# ── 4. G1 MuJoCo モデルを取得 (MuJoCo Menagerie) ─────────────────────────
print('\nG1 モデルを取得中 (MuJoCo Menagerie)...')
if not os.path.exists('/content/mujoco_menagerie/.git'):
    subprocess.run([
        'git', 'clone', '--depth=1', '--filter=blob:none', '--sparse',
        'https://github.com/google-deepmind/mujoco_menagerie.git',
        '/content/mujoco_menagerie'
    ], check=True)
    subprocess.run([
        'git', '-C', '/content/mujoco_menagerie',
        'sparse-checkout', 'set', 'unitree_g1'
    ], check=True)
else:
    print('  スキップ (既存)')

G1_SCENE_XML = '/content/mujoco_menagerie/unitree_g1/scene.xml'
assert os.path.exists(G1_SCENE_XML), f'G1 モデルが見つかりません: {G1_SCENE_XML}'
print('\n✅ セットアップ完了！次のセルへ進んでください。')

In [ ]:
#@title 📁 チーム名と Google Drive 接続

#@markdown チーム名を入力してください（モデルの保存先になります）
team_name = '自分のチーム名' #@param {type:"string"}

from google.colab import drive
print('Google Drive に接続します...')
drive.mount('/content/drive')

import os
SAVE_DIR = f'/content/drive/MyDrive/RoboQuest2026/{team_name}/g1'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'\n✅ 保存先: {SAVE_DIR}')

## 📚 G1 ロボットを知ろう

### 関節構造（29 DOF）

```
G1 関節マップ
─────────────────────────────────────────
【下半身】左脚 6 関節 + 右脚 6 関節 = 12
  hip_pitch  ← 前後の振り脚（歩行の主役）
  hip_roll   ← 左右の安定
  hip_yaw    ← 水平回転
  knee       ← 膝の曲げ伸ばし
  ankle_pitch← つま先の上下
  ankle_roll ← 足首の内外傾き

【腰】3 関節
  waist_yaw / waist_roll / waist_pitch

【上半身】左腕 7 関節 + 右腕 7 関節 = 14
  shoulder_pitch/roll/yaw, elbow
  wrist_roll/pitch/yaw
─────────────────────────────────────────
合計: 12 + 3 + 14 = 29 DOF
```

### 観測空間（98 次元）

| インデックス | 内容 | 次元数 |
|-------------|------|-------|
| [0:3]       | 速度コマンド (vx, vy, ω) | 3 |
| [3:6]       | 胴体の角速度 xyz | 3 |
| [6:9]       | 重力ベクトル（ボディフレーム） | 3 |
| [9:11]      | **歩行位相 (sin, cos)** ← 交互歩行の鍵！ | 2 |
| [11:40]     | 関節角度（立位からの差） | 29 |
| [40:69]     | 関節の角速度 | 29 |
| [69:98]     | 前回の行動 | 29 |

### 💡 歩行位相って何？

二足歩行は「左足・右足を交互に出す」リズムが不可欠です。  
位相信号 `[sin(2πt/T), cos(2πt/T)]` を観測に加えることで、
AI がどのタイミングでどちらの足を上げるべきかを学びやすくなります。  
（参考: [unitree_rl_mjlab `phase()` 関数](https://github.com/unitreerobotics/unitree_rl_mjlab)）

In [ ]:
#@title 🤖 G1 環境クラス（unitree_rl_mjlab 完全準拠 — 変更不要）
# ============================================================
#  G1WalkEnv — unitree_rl_mjlab velocity_env_cfg.py 完全準拠
#
#  ✅ 終了判定: 傾き>70° (bad_orientation) ← 高さ判定ではない!
#  ✅ 終了ペナルティ: -200.0
#  ✅ variable_posture: 速度依存・関節別 std
#  ✅ feet_gait: offset=[0.0, 0.5] 左右交互 period=0.6s thr=0.56
#  ✅ foot_slip / foot_clearance / stand_still / joint_acc_l2
# ============================================================

import math
import mujoco
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from typing import Optional

G1_STANDING_CTRL = np.array([
    0.0,  0.0, 0.0, 0.0, 0.0, 0.0,
    0.0,  0.0, 0.0, 0.0, 0.0, 0.0,
    0.0,  0.0, 0.0,
    0.2,  0.2, 0.0, 1.28, 0.0, 0.0, 0.0,
    0.2, -0.2, 0.0, 1.28, 0.0, 0.0, 0.0,
], dtype=np.float64)

G1_ACTION_SCALE = np.array([
    0.40, 0.40, 0.30, 0.40, 0.25, 0.20,
    0.40, 0.40, 0.30, 0.40, 0.25, 0.20,
    0.20, 0.15, 0.15,
    0.30, 0.25, 0.25, 0.25, 0.15, 0.10, 0.10,
    0.30, 0.25, 0.25, 0.25, 0.15, 0.10, 0.10,
], dtype=np.float64)

_STD_STANDING = np.full(29, 0.05, dtype=np.float64)

_STD_WALKING = np.array([
    0.50, 0.15, 0.15, 0.50, 0.15, 0.10,
    0.50, 0.15, 0.15, 0.50, 0.15, 0.10,
    0.15, 0.10, 0.10,
    0.15, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10,
    0.15, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10,
], dtype=np.float64)

_STD_RUNNING = np.array([
    0.50, 0.25, 0.25, 0.50, 0.25, 0.10,
    0.50, 0.25, 0.25, 0.50, 0.25, 0.10,
    0.25, 0.10, 0.10,
    0.25, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10,
    0.25, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10,
], dtype=np.float64)

G1_N_JOINTS = 29
G1_CTRL_DT  = 0.02
G1_SUBSTEPS = 4
FOOT_BODY_NAMES = ["left_ankle_roll_link", "right_ankle_roll_link"]
GAIT_PERIOD  = 0.6
GAIT_OFFSETS = [0.0, 0.5]
GAIT_THR     = 0.56


class G1WalkEnv(gym.Env):
    """G1 29-DOF 二足歩行環境 (unitree_rl_mjlab 完全準拠).

    観測 (98 次元): cmd(3)+ang_vel(3)+proj_grav(3)+phase(2)+jpos(29)+jvel(29)+last_act(29)
    行動 (29 次元): 各関節オフセット [-1, 1]
    終了: 傾き > 70° (bad_orientation)
    """

    metadata = {"render_modes": ["rgb_array"], "render_fps": 50}

    def __init__(
        self,
        max_episode_steps: int = 1000,
        render_mode=None,
        xml_path=None,
        randomize_cmd: bool = True,
        cmd_scale: float = 1.0,
    ):
        super().__init__()
        self.max_episode_steps = max_episode_steps
        self.render_mode = render_mode
        self.randomize_cmd = randomize_cmd
        self.cmd_scale = cmd_scale

        xml = xml_path or G1_SCENE_XML
        self.model = mujoco.MjModel.from_xml_path(xml)
        self.data  = mujoco.MjData(self.model)

        self._key_id = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_KEY, "stand")

        self._foot_body_ids = [
            bid for bid in [
                mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_BODY, n)
                for n in FOOT_BODY_NAMES
            ] if bid >= 0
        ]

        jnt_range = self.model.jnt_range[1:1 + G1_N_JOINTS]
        margin = 0.1 * (jnt_range[:, 1] - jnt_range[:, 0])
        self._soft_lower = jnt_range[:, 0] + margin
        self._soft_upper = jnt_range[:, 1] - margin

        obs_dim = 3 + 3 + 3 + 2 + G1_N_JOINTS * 3
        self.observation_space = spaces.Box(-np.inf, np.inf, (obs_dim,), np.float32)
        self.action_space      = spaces.Box(-1.0, 1.0, (G1_N_JOINTS,), np.float32)

        self._step_count  = 0
        self._last_action = np.zeros(G1_N_JOINTS)
        self._last_jvel   = np.zeros(G1_N_JOINTS)
        self._vel_cmd     = np.zeros(3)

        if render_mode == "rgb_array":
            self._renderer = mujoco.Renderer(self.model, height=480, width=640)
        else:
            self._renderer = None

    def set_vel_cmd(self, vx, vy, omega):
        self._vel_cmd = np.array([vx, vy, omega])

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        mujoco.mj_resetData(self.model, self.data)
        if self._key_id >= 0:
            mujoco.mj_resetDataKeyframe(self.model, self.data, self._key_id)
        self.data.qpos[7:] += self.np_random.uniform(-0.05, 0.05, G1_N_JOINTS)
        self.data.qvel[:] = 0.0
        mujoco.mj_forward(self.model, self.data)
        if self.randomize_cmd:
            s = self.cmd_scale
            self._vel_cmd = np.array([
                self.np_random.uniform(-0.5 * s, 1.0 * s),
                self.np_random.uniform(-0.5 * s, 0.5 * s),
                self.np_random.uniform(-1.0 * s, 1.0 * s),
            ])
        self._step_count  = 0
        self._last_action = np.zeros(G1_N_JOINTS)
        self._last_jvel   = self.data.qvel[6:6 + G1_N_JOINTS].copy()
        return self._get_obs().astype(np.float32), {}

    def step(self, action):
        action = np.clip(action, -1.0, 1.0)
        q_target = G1_STANDING_CTRL + action * G1_ACTION_SCALE
        q_target = np.clip(q_target,
                           self.model.actuator_ctrlrange[:, 0],
                           self.model.actuator_ctrlrange[:, 1])
        self.data.ctrl[:] = q_target
        jvel_before = self.data.qvel[6:6 + G1_N_JOINTS].copy()
        for _ in range(G1_SUBSTEPS):
            mujoco.mj_step(self.model, self.data)
        self._step_count += 1
        terminated = self._is_terminated()
        obs    = self._get_obs().astype(np.float32)
        reward = self._compute_reward(action, terminated, jvel_before)
        truncated = self._step_count >= self.max_episode_steps
        self._last_action = action.copy()
        self._last_jvel   = self.data.qvel[6:6 + G1_N_JOINTS].copy()
        return obs, reward, terminated, truncated, {}

    def render(self):
        if self._renderer is None:
            return None
        self._renderer.update_scene(self.data)
        return self._renderer.render()

    def close(self):
        if self._renderer is not None:
            self._renderer.close()

    def _get_obs(self):
        ang_vel   = self.data.qvel[3:6].copy()
        proj_grav = self._projected_gravity()
        t   = self._step_count * G1_CTRL_DT
        phi = (t % GAIT_PERIOD) / GAIT_PERIOD
        phase = np.array([math.sin(2 * math.pi * phi),
                          math.cos(2 * math.pi * phi)])
        jpos = self.data.qpos[7:7 + G1_N_JOINTS] - G1_STANDING_CTRL
        jvel = self.data.qvel[6:6 + G1_N_JOINTS].copy()
        return np.concatenate([self._vel_cmd, ang_vel, proj_grav, phase,
                               jpos, jvel, self._last_action])

    def _projected_gravity(self):
        qw, qx, qy, qz = self.data.qpos[3:7]
        R = np.array([
            [1-2*(qy**2+qz**2), 2*(qx*qy+qw*qz),   2*(qx*qz-qw*qy)],
            [2*(qx*qy-qw*qz),   1-2*(qx**2+qz**2),  2*(qy*qz+qw*qx)],
            [2*(qx*qz+qw*qy),   2*(qy*qz-qw*qx),   1-2*(qx**2+qy**2)],
        ])
        return R.T @ np.array([0.0, 0.0, -1.0])

    def _is_terminated(self):
        # bad_orientation: 傾き > 70°
        proj_grav = self._projected_gravity()
        return bool(-proj_grav[2] < math.cos(math.radians(70.0)))

    def _compute_reward(self, action, terminated, jvel_before):
        cmd       = self._vel_cmd
        # qvel[:3] は world フレーム → body フレームに変換 (root_lin_vel_b)
        # qvel[3:6] は MuJoCo 慣例で body フレーム角速度
        ang_vel   = self.data.qvel[3:6]
        proj_grav = self._projected_gravity()
        jpos      = self.data.qpos[7:7 + G1_N_JOINTS]
        jvel      = self.data.qvel[6:6 + G1_N_JOINTS]

        # body フレーム線速度 (root_lin_vel_b): R^T @ qvel_world
        qw, qx, qy, qz = self.data.qpos[3:7]
        _R = np.array([
            [1-2*(qy**2+qz**2), 2*(qx*qy+qw*qz),   2*(qx*qz-qw*qy)],
            [2*(qx*qy-qw*qz),   1-2*(qx**2+qz**2),  2*(qy*qz+qw*qx)],
            [2*(qx*qz+qw*qy),   2*(qy*qz-qw*qx),   1-2*(qx**2+qy**2)],
        ])
        lin_vel_b = _R.T @ self.data.qvel[:3]

        # 1. track_linear_velocity  w=1.0  std2=0.25 (unitree_rl_mjlab: root_lin_vel_b)
        xy_err = float(np.sum((cmd[:2] - lin_vel_b[:2]) ** 2))
        z_err  = float(lin_vel_b[2] ** 2)
        r_lin  = math.exp(-(xy_err + 2.0 * z_err) / 0.25)

        # 2. track_angular_velocity  w=1.0  std2=0.5
        z_ang_err  = float((cmd[2] - ang_vel[2]) ** 2)
        xy_ang_err = float(np.sum(ang_vel[:2] ** 2))
        r_ang = math.exp(-(z_ang_err + 0.05 * xy_ang_err) / 0.5)

        # 3. body_orientation_l2  w=-1.0
        r_orient = -1.0 * float(np.sum(proj_grav[:2] ** 2))

        # 4. variable_posture  w=1.0  速度依存・関節別 std
        speed = float(np.linalg.norm(cmd[:2])) + abs(float(cmd[2]))
        std = _STD_STANDING if speed < 0.1 else (_STD_WALKING if speed < 1.5 else _STD_RUNNING)
        r_posture = float(np.exp(-np.mean((jpos - G1_STANDING_CTRL) ** 2 / std ** 2)))

        # 5. body_ang_vel  w=-0.05
        r_ang_vel = -0.05 * float(np.sum(ang_vel[:2] ** 2))

        # 6. is_terminated  w=-200.0
        r_term = -200.0 if terminated else 0.0

        # 7. joint_acc_l2  w=-2.5e-7
        jacc = (jvel - jvel_before) / G1_CTRL_DT
        r_jacc = -2.5e-7 * float(np.sum(jacc ** 2))

        # 8. joint_pos_limits  w=-10.0  (soft 90%)
        viol = (np.maximum(0.0, self._soft_lower - jpos)
              + np.maximum(0.0, jpos - self._soft_upper))
        r_limits = -10.0 * float(np.sum(viol))

        # 9. action_rate_l2  w=-0.05
        r_rate = -0.05 * float(np.sum((action - self._last_action) ** 2))

        # 10. feet_gait  w=0.5
        r_gait = self._feet_gait_reward(speed)

        # 11&12. foot_clearance / foot_slip
        r_clear, r_slip = self._feet_rewards(speed)

        # 13. stand_still  w=-1.0
        r_still = (-1.0 * float(np.sum((jpos - G1_STANDING_CTRL) ** 2))
                   if speed <= 0.1 else 0.0)

        return (r_lin + r_ang + r_orient + r_posture + r_ang_vel
                + r_term + r_jacc + r_limits + r_rate
                + r_gait + r_clear + r_slip + r_still)

    def _feet_gait_reward(self, speed):
        if speed <= 0.1 or not self._foot_body_ids:
            return 0.0
        g_phase = ((self._step_count * G1_CTRL_DT) / GAIT_PERIOD) % 1.0
        contacts = [float(self.data.xpos[bid][2]) < 0.10 for bid in self._foot_body_ids]
        score = sum(
            1.0 if (((g_phase + off) % 1.0) < GAIT_THR) == c else 0.0
            for off, c in zip(GAIT_OFFSETS, contacts)
        )
        return 0.5 * score / len(GAIT_OFFSETS)

    def _feet_rewards(self, speed):
        if speed <= 0.1 or not self._foot_body_ids:
            return 0.0, 0.0
        r_clear = r_slip = 0.0
        buf = np.zeros(6)
        for bid in self._foot_body_ids:
            foot_z = float(self.data.xpos[bid][2])
            mujoco.mj_objectVelocity(
                self.model, self.data, mujoco.mjtObj.mjOBJ_BODY, bid, buf, 0)
            fv_xy = buf[3:5]
            r_clear += -1.0 * abs(foot_z - 0.10) * float(np.linalg.norm(fv_xy))
            if foot_z < 0.10:
                r_slip += -0.25 * float(np.sum(fv_xy ** 2))
        return r_clear, r_slip


print("✅ G1WalkEnv (unitree_rl_mjlab 完全準拠) 定義完了")
print()
print("報酬構成 (velocity_env_cfg.py + g1/env_cfgs.py 準拠):")
for n, w, d in [
    ("track_lin_vel  ", " 1.0 ", "exp(-err/0.25)  xy速度 + 2z速度"),
    ("track_ang_vel  ", " 1.0 ", "exp(-err/0.5)   z角速度 + 0.05xy角速度"),
    ("orientation    ", "-1.0 ", "傾きペナルティ"),
    ("posture        ", " 1.0 ", "速度依存・関節別std exp(-mean(err/std)²)"),
    ("body_ang_vel   ", "-0.05", "胴体角速度 xy"),
    ("is_terminated  ", "-200 ", "転倒ペナルティ ← 最重要"),
    ("joint_acc      ", "-2.5e-7","関節加速度"),
    ("joint_limits   ", "-10  ", "soft limit 違反"),
    ("action_rate    ", "-0.05", "行動変化"),
    ("feet_gait      ", " 0.5 ", "位相交互歩行 offset=[0, 0.5]"),
    ("foot_clearance ", "-1.0 ", "足高さ × 足速度"),
    ("foot_slip      ", "-0.25", "接地中の横方向速度"),
    ("stand_still    ", "-1.0 ", "ゼロcmd時の姿勢ずれ"),
]:
    print(f"  {n} w={w}  {d}")
print()
print("終了条件: 傾き > 70° (bad_orientation)")


In [ ]:
#@title 🔍 環境を確認する
import mujoco, numpy as np

env = G1WalkEnv()
obs, _ = env.reset()

print("=" * 55)
print("G1WalkEnv (unitree_rl_mjlab 準拠版) 基本情報")
print("=" * 55)
print(f"観測空間: {env.observation_space.shape[0]} 次元")
print(f"行動空間: {env.action_space.shape[0]} 関節")
print(f"初期高さ: {env.data.qpos[2]:.3f} m")
print(f"足ボディ検出: {len(env._foot_body_ids)} 個")
print()

groups = {"左脚": range(0, 6), "右脚": range(6, 12), "腰  ": range(12, 15),
          "左腕": range(15, 22), "右腕": range(22, 29)}
for grp, ids in groups.items():
    names = [mujoco.mj_id2name(env.model, mujoco.mjtObj.mjOBJ_ACTUATOR, i) for i in ids]
    print(f"  [{grp}] {names}")

obs2, r, term, _, _ = env.step(np.zeros(29))
print(f"\n1ステップ後 — 報酬: {r:.4f}, 終了: {term}")
print(f"  高さ: {env.data.qpos[2]:.3f} m")
env.close()
print("\n✅ 環境確認完了！")


## 🎛️ フェーズ1: コマンド速度の設定

G1 の報酬関数は **unitree_rl_mjlab の velocity_env_cfg.py をそのまま移植** しています。

### 報酬内訳（固定）

| 報酬項目 | 重み | 説明 |
|---------|------|------|
| track_lin_vel | +1.0 | xy速度 + 2×z速度の追跡 |
| track_ang_vel | +1.0 | z角速度 + 0.05×xy角速度 |
| body_orientation | -1.0 | 胴体の傾きペナルティ |
| **variable_posture** | **+1.0** | **速度依存・関節別 std で姿勢を維持 ← 安定の鍵** |
| body_ang_vel | -0.05 | 胴体角速度 xy |
| **is_terminated** | **-200** | **転倒ペナルティ ← 最重要** |
| joint_acc | -2.5e-7 | 関節加速度 |
| joint_limits | -10 | 関節可動域超過 |
| action_rate | -0.05 | 急激な行動変化 |
| **feet_gait** | **+0.5** | **左右交互歩行リズム** |
| foot_clearance | -1.0 | 足の高さ × 速度 |
| foot_slip | -0.25 | 接地中の横スリップ |
| stand_still | -1.0 | ゼロcmd時の静止 |

### ❗ 元の設計との主な違い（以前のバージョンからの改善）

| 項目 | 以前 | 今回 |
|-----|------|------|
| 終了条件 | 高さ < 0.5m | **傾き > 70° (bad_orientation)** |
| 終了ペナルティ | -20 | **-200** |
| 姿勢報酬 | 高さ偏差のみ | **variable_posture（速度依存）** |
| 歩行リズム | 簡易位相 | **offset=[0, 0.5] 左右交互** |

調整できるのは **コマンド速度スケール** のみです。→ 次のセルへ


In [ ]:
#@title 🎛️ コマンド速度スケールの設定
#@markdown 報酬は **unitree_rl_mjlab 完全準拠** のため固定です。
#@markdown
#@markdown - **0.5 以下**: 遅い歩行から学習（初期推奨）
#@markdown - **1.0** (推奨): 原典と同じ速度範囲
#@markdown - **1.5 以上**: 速い歩行（難易度高）

cmd_scale = 1.0  #@param {type:"slider", min:0.3, max:2.0, step:0.1}

print(f"✅ コマンド速度スケール: {cmd_scale}")
print(f"  vx 範囲: [{-0.5*cmd_scale:.2f}, {1.0*cmd_scale:.2f}] m/s")
print(f"  vy 範囲: [{-0.5*cmd_scale:.2f}, {0.5*cmd_scale:.2f}] m/s")
print(f"  omega 範囲: [{-1.0*cmd_scale:.2f}, {1.0*cmd_scale:.2f}] rad/s")


In [ ]:
#@title ⚙️ 学習設定

#@markdown ### 学習規模
#@markdown **学習ステップ数**（多いほど上手になるが時間がかかる）
total_timesteps = 500000 #@param {type:"slider", min:100000, max:2000000, step:100000}

#@markdown **並列環境数**（多いほど学習が速い — メモリと相談）
num_envs = 4 #@param {type:"slider", min:1, max:8, step:1}

#@markdown ---
#@markdown ### PPO ハイパーパラメータ
#@markdown **学習率**（通常は変えなくてOK）
learning_rate = 0.0003 #@param {type:"slider", min:0.0001, max:0.001, step:0.0001}

#@markdown **1回の更新で集めるステップ数**
n_steps = 2048 #@param {type:"slider", min:512, max:4096, step:256}

#@markdown **ミニバッチサイズ**
batch_size = 512 #@param {type:"slider", min:128, max:1024, step:128}

#@markdown ---
#@markdown ### ネットワーク構造
#@markdown **隠れ層のサイズ**（G1 は Go2 より大きいネットが必要）
net_arch_choice = '256-256-128 (推奨)' #@param ["512-256-128 (大)", "256-256-128 (推奨)", "128-128 (小)"]

arch_map = {
    '512-256-128 (大)': [512, 256, 128],
    '256-256-128 (推奨)': [256, 256, 128],
    '128-128 (小)': [128, 128],
}
net_arch = arch_map[net_arch_choice]

est_minutes_low  = total_timesteps // 100000 * 3
est_minutes_high = total_timesteps // 100000 * 10
print(f'学習設定:')
print(f'  ステップ数: {total_timesteps:,}')
print(f'  並列環境: {num_envs}')
print(f'  ネット構造: {net_arch}')
print(f'  ⏱ 目安: 約 {est_minutes_low}〜{est_minutes_high} 分 (GPU 有無で大きく変わります)')

In [ ]:
#@title 🚀 G1 歩行学習スタート！

import os, sys
os.chdir('/content/RoboQuest2026')
if '/content/RoboQuest2026' not in sys.path:
    sys.path.insert(0, '/content/RoboQuest2026')

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.callbacks import CheckpointCallback

# monitor_dir を使うと環境ごとに別ファイル (0.monitor.csv, 1.monitor.csv ...) が作成される
MONITOR_DIR = '/tmp/g1_monitors'
os.makedirs(MONITOR_DIR, exist_ok=True)

def _make_g1_env():
    return G1WalkEnv(
        cmd_scale=cmd_scale,
        max_episode_steps=1000,
    )

print('環境を作成中...')
vec_env = make_vec_env(_make_g1_env, n_envs=num_envs, monitor_dir=MONITOR_DIR)
vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True)

model = PPO(
    'MlpPolicy',
    vec_env,
    learning_rate=learning_rate,
    n_steps=n_steps,
    batch_size=batch_size,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    policy_kwargs={'net_arch': net_arch},
    verbose=0,
)

total_params = sum(p.numel() for p in model.policy.parameters())
print(f'モデルパラメータ数: {total_params:,}')
print(f'
🦿 G1 歩行学習を開始します... ({total_timesteps:,} ステップ)')

checkpoint_cb = CheckpointCallback(
    save_freq=max(50000 // num_envs, 1),
    save_path='/tmp/g1_checkpoints/',
    name_prefix='g1_walk',
)

model.learn(
    total_timesteps=total_timesteps,
    callback=checkpoint_cb,
    progress_bar=True,
)

G1_MODEL_PATH = '/tmp/g1_walk_model'
G1_VECNORM_PATH = '/tmp/g1_walk_vecnorm.pkl'
model.save(G1_MODEL_PATH)
vec_env.save(G1_VECNORM_PATH)

try:
    model.save(f'{SAVE_DIR}/g1_walk_model')
    vec_env.save(f'{SAVE_DIR}/g1_walk_vecnorm.pkl')
    print(f'
💾 Google Drive に保存: {SAVE_DIR}')
except Exception:
    pass

print('
✅ 学習完了！次のセルで学習曲線と動画を確認しましょう。')


In [ ]:
#@title 📈 学習曲線を確認しよう

import glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['IPAexGothic', 'Noto Sans CJK JP', 'DejaVu Sans']

MONITOR_DIR = '/tmp/g1_monitors'
monitor_files = glob.glob(f'{MONITOR_DIR}/*.monitor.csv')

if not monitor_files:
    print('⚠️ モニターファイルが見つかりません。学習を先に実行してください。')
    print(f'検索先: {MONITOR_DIR}/*.monitor.csv')
else:
    print(f'モニターファイル: {len(monitor_files)} 件')
    dfs = []
    for f in monitor_files:
        try:
            df = pd.read_csv(f, skiprows=1)
            dfs.append(df)
        except Exception as e:
            print(f'  ⚠️ 読み込み失敗: {f} → {e}')

    if not dfs:
        print('❌ データが読み込めませんでした')
    else:
        data = pd.concat(dfs).sort_values('t').reset_index(drop=True)
        data['episode'] = range(len(data))

        window = max(1, len(data) // 50)
        data['reward_smooth'] = data['r'].rolling(window=window, min_periods=1).mean()
        data['length_smooth'] = data['l'].rolling(window=window, min_periods=1).mean()

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        ax1.plot(data['episode'], data['r'], alpha=0.2, color='steelblue', linewidth=0.5)
        ax1.plot(data['episode'], data['reward_smooth'], color='steelblue', linewidth=2)
        ax1.set_xlabel('エピソード')
        ax1.set_ylabel('エピソード報酬')
        ax1.set_title('G1 歩行 学習曲線（報酬）')
        ax1.grid(True, alpha=0.3)

        ax2.plot(data['episode'], data['l'], alpha=0.2, color='darkorange', linewidth=0.5)
        ax2.plot(data['episode'], data['length_smooth'], color='darkorange', linewidth=2)
        ax2.set_xlabel('エピソード')
        ax2.set_ylabel('エピソード長さ [ステップ]')
        ax2.set_title('G1 歩行 学習曲線（生存ステップ数）')
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig('/tmp/g1_learning_curve.png', dpi=120, bbox_inches='tight')
        plt.show()

        n_ep = len(data)
        last_n = data.tail(max(1, n_ep // 5))
        print(f'
学習サマリー:')
        print(f'  総エピソード数: {n_ep}')
        print(f'  最終期平均報酬: {last_n["r"].mean():.2f} ± {last_n["r"].std():.2f}')
        print(f'  最終期平均生存: {last_n["l"].mean():.1f} ステップ')


In [ ]:
#@title 🎬 G1 の歩行を動画で確認しよう

import mujoco
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecNormalize, DummyVecEnv

print('学習済みモデルを読み込み中...')
loaded_model = PPO.load(G1_MODEL_PATH)

# VecNormalize の統計を再現するためにダミー env を作成
_dummy_env = DummyVecEnv([lambda: G1WalkEnv(reward_config=g1_reward_cfg)])
eval_env = VecNormalize.load(G1_VECNORM_PATH, _dummy_env)
eval_env.training = False
eval_env.norm_reward = False

# 録画
raw_env = G1WalkEnv(
    reward_config=g1_reward_cfg,
    max_episode_steps=600,
    render_mode='rgb_array',
)
renderer = mujoco.Renderer(raw_env.model, height=480, width=640)
renderer.scene.flags[mujoco.mjtRndFlag.mjRND_SHADOW] = False

frames = []
heights = []
rewards_log = []

# 前向き歩行のコマンドで評価
obs_raw, _ = raw_env.reset()
raw_env.set_vel_cmd(vx=0.5, vy=0.0, omega=0.0)
raw_env.randomize_cmd = False

# eval_env の正規化統計を使って obs を変換
obs_vec = eval_env.normalize_obs(obs_raw[np.newaxis, :])

for step in range(600):
    action, _ = loaded_model.predict(obs_vec, deterministic=True)
    obs_raw, r, terminated, truncated, _ = raw_env.step(action[0])
    obs_vec = eval_env.normalize_obs(obs_raw[np.newaxis, :])

    heights.append(float(raw_env.data.qpos[2]))
    rewards_log.append(r)

    renderer.update_scene(raw_env.data)
    frames.append(renderer.render().copy())

    if terminated or truncated:
        print(f'  エピソード終了: {step + 1} ステップ')
        break

renderer.close()
raw_env.close()
eval_env.close()

print(f'\n録画フレーム数: {len(frames)}')
print(f'平均重心高さ: {np.mean(heights):.3f} m  (立位: {G1_STANDING_H} m)')
print(f'平均報酬/ステップ: {np.mean(rewards_log):.4f}')

# 動画表示
try:
    import mediapy as media
    media.show_video(frames, fps=30)
except ImportError:
    from matplotlib import animation
    from IPython.display import HTML, display
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.axis('off')
    img = ax.imshow(frames[0])
    ani = animation.FuncAnimation(
        fig, lambda f: [img.set_array(f)], frames=frames, interval=33, blit=True)
    plt.close(fig)
    display(HTML(ani.to_jshtml()))

# 重心高さの推移をプロット
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(heights, color='steelblue')
ax.axhline(G1_STANDING_H, color='green', linestyle='--', label=f'立位高さ ({G1_STANDING_H} m)')
ax.axhline(G1_FALL_H, color='red', linestyle='--', label=f'転倒閾値 ({G1_FALL_H} m)')
ax.set_xlabel('ステップ')
ax.set_ylabel('重心高さ [m]')
ax.set_title('G1 重心高さの推移')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#@title 🔬 関節ごとの動きを分析しよう

# 各関節がどのくらい動いているか可視化
import mujoco
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecNormalize, DummyVecEnv

loaded_model = PPO.load(G1_MODEL_PATH)
_dummy_env = DummyVecEnv([lambda: G1WalkEnv(reward_config=g1_reward_cfg)])
eval_env = VecNormalize.load(G1_VECNORM_PATH, _dummy_env)
eval_env.training = False
eval_env.norm_reward = False

raw_env = G1WalkEnv(reward_config=g1_reward_cfg, max_episode_steps=300)
obs_raw, _ = raw_env.reset()
raw_env.set_vel_cmd(vx=0.5, vy=0.0, omega=0.0)
raw_env.randomize_cmd = False

obs_vec = eval_env.normalize_obs(obs_raw[np.newaxis, :])
joint_traj = []

for _ in range(300):
    action, _ = loaded_model.predict(obs_vec, deterministic=True)
    obs_raw, _, terminated, truncated, _ = raw_env.step(action[0])
    obs_vec = eval_env.normalize_obs(obs_raw[np.newaxis, :])
    joint_traj.append(raw_env.data.qpos[7:7+G1_N_JOINTS].copy())
    if terminated or truncated:
        break

raw_env.close()
eval_env.close()

joint_traj = np.array(joint_traj)  # (T, 29)
T = len(joint_traj)
t_axis = np.arange(T) * G1_CTRL_DT

# 下半身（脚）の主要関節を表示
key_joints = {
    '左 hip_pitch': 0, '右 hip_pitch': 6,
    '左 knee': 3, '右 knee': 9,
    '左 ankle_pitch': 4, '右 ankle_pitch': 10,
}

fig, axes = plt.subplots(3, 2, figsize=(14, 9))
fig.suptitle('G1 脚の関節角度軌跡', fontsize=13)

for ax, (name, idx) in zip(axes.flatten(), key_joints.items()):
    ax.plot(t_axis, joint_traj[:, idx], linewidth=1.5)
    ax.axhline(G1_STANDING_CTRL[idx], color='gray', linestyle='--', alpha=0.6, label='立位')
    ax.set_xlabel('時間 [s]')
    ax.set_ylabel('角度 [rad]')
    ax.set_title(name)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/g1_joint_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ 関節分析完了')

In [ ]:
#@title 💾 モデルを保存して提出

import os, json

params = {
    'team_name': team_name,
    'robot': 'G1_29DOF',
    'task': 'velocity_tracking',
    'framework': 'stable-baselines3 PPO',
    'reward': {
        'framework': 'unitree_rl_mjlab velocity_env_cfg.py 準拠 (固定)',
        'cmd_scale': cmd_scale,
    },
    'training': {
        'total_timesteps': total_timesteps,
        'num_envs': num_envs,
        'learning_rate': learning_rate,
        'n_steps': n_steps,
        'batch_size': batch_size,
        'net_arch': net_arch,
    },
    'reference': 'https://github.com/unitreerobotics/unitree_rl_mjlab',
}

try:
    os.makedirs(SAVE_DIR, exist_ok=True)
    with open(f'{SAVE_DIR}/params.json', 'w', encoding='utf-8') as f:
        json.dump(params, f, ensure_ascii=False, indent=2)

    import shutil
    for fname in ['g1_learning_curve.png', 'g1_joint_analysis.png']:
        src = f'/tmp/{fname}'
        if os.path.exists(src):
            shutil.copy(src, f'{SAVE_DIR}/{fname}')

    print(f'✅ 保存完了！')
    print(f'  モデル:    {SAVE_DIR}/g1_walk_model.zip')
    print(f'  正規化統計: {SAVE_DIR}/g1_walk_vecnorm.pkl')
    print(f'  パラメータ: {SAVE_DIR}/params.json')
    print(f'  グラフ:    {SAVE_DIR}/g1_*.png')
    print(f'\n🎉 G1 歩行学習ノートブック 完了！')

except Exception as e:
    print(f'Drive 保存エラー: {e}')
    print('ローカル保存済み: /tmp/g1_walk_model.zip')